In [0]:
# This notebook requires a Kaggle API Token, defined in 'kaggle' databricks secrets scope.
# Run the following Databricks CLI commands in a safe environment to create the scope and secret:
# databricks secrets create-scope kaggle
# databricks secrets put-secret kaggle KAGGLE_API_TOKEN --string-value "<your_token>"

In [0]:
import os

import kagglehub
from kagglehub import KaggleDatasetAdapter

from olist_lakehouse.transformations import add_metadata

In [0]:
try:
    os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kaggle", "KAGGLE_API_TOKEN")
except Exception as e:
    raise ValueError("Could not load Kaggle API Token, be sure to follow the steps at the beginning of this notebook to set the secrets!")

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
dataset = dbutils.widgets.get("dataset")

table_output_name = f"{catalog}.{schema}.tbl_{dataset.replace('.csv', '')}"

In [0]:
print("-" * 50)
print("Starting dataset download: ", dataset)
pandas_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    dataset,
)
print("Downloaded successfully!")
print("Adding metadata...")
spark_df = (
    spark.createDataFrame(pandas_df)
    .transform(add_metadata)
)
print("Writing to catalog delta table: ", table_output_name)
spark_df.write.mode("overwrite").saveAsTable(table_output_name)
print("Done!!!")
print("-" * 50)